# Lab 3 - HiNetLite NPU v6

- Model: `v5_clean_no_block_residual`
- Goal: continue the v5 low-risk Mobilint NPU path with fewer internal add operations.
- Expected input/output: `256x256x3` RGB.
- Submission path: this notebook is self-contained and does not import repo-local helpers.


## 1. Setup

Imports, Google Drive paths, run configuration, and artifact directories.

In [1]:
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    %pip install -q onnx onnxruntime
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
from __future__ import annotations

import csv
import json
import math
import os
import random
import shlex
import shutil
import subprocess
import sys
import time
from collections import Counter
from contextlib import nullcontext
from dataclasses import asdict, dataclass
from datetime import datetime
from pathlib import Path
from typing import Any

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset

try:
    import onnx
except ImportError:
    onnx = None

try:
    import onnxruntime as ort
except ImportError:
    ort = None

try:
    BICUBIC_RESAMPLE = Image.Resampling.BICUBIC
except AttributeError:
    BICUBIC_RESAMPLE = Image.BICUBIC

try:
    FLIP_LEFT_RIGHT = Image.Transpose.FLIP_LEFT_RIGHT
    FLIP_TOP_BOTTOM = Image.Transpose.FLIP_TOP_BOTTOM
    ROTATE_90 = Image.Transpose.ROTATE_90
    ROTATE_180 = Image.Transpose.ROTATE_180
    ROTATE_270 = Image.Transpose.ROTATE_270
except AttributeError:
    FLIP_LEFT_RIGHT = Image.FLIP_LEFT_RIGHT
    FLIP_TOP_BOTTOM = Image.FLIP_TOP_BOTTOM
    ROTATE_90 = Image.ROTATE_90
    ROTATE_180 = Image.ROTATE_180
    ROTATE_270 = Image.ROTATE_270

def set_global_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def now_stamp() -> str:
    return datetime.now().strftime('%H%M%S_%d%m')

def make_unique_run_root(runs_root: Path, run_slug: str) -> Path:
    safe_slug = ''.join(ch if ch.isalnum() or ch in '_-' else '_' for ch in run_slug).strip('_')
    candidate = runs_root / f'{now_stamp()}_{safe_slug}'
    while candidate.exists():
        time.sleep(1.1)
        candidate = runs_root / f'{now_stamp()}_{safe_slug}'
    return candidate

DRIVE_ROOT = Path('/content/drive/MyDrive/Data 255 Class Spring 2026/Data 255/Lab 3')
DRIVE_DATA_ROOT = DRIVE_ROOT / 'Data'
DRIVE_RUNS_ROOT = DRIVE_ROOT / 'runs'
IMAGE_SUFFIXES = {'.png', '.jpg', '.jpeg', '.bmp'}

@dataclass
class RunConfig:
    model_id: str = 'hinet_lite_npu_v6_v5_clean_no_block_residual'
    run_slug: str = 'v5_clean_no_block_residual'
    seed: int = 1337
    eval_size: int = 256
    crop_size: int = 128
    batch_size: int = 8
    epochs: int = 120
    lr: float = 2.5e-4
    min_lr: float = 1e-5
    weight_decay: float = 5e-5
    warmup_epochs: int = 5
    grad_clip_norm: float = 1.0
    mse_weight: float = 1.0
    l1_weight: float = 0.1
    num_workers: int = 2
    use_amp: bool = True
    use_ema: bool = True
    ema_decay: float = 0.999
    selected_weight_source: str = 'ema'
    train_pair_limit: int | None = None
    val_pair_limit: int | None = None
    calibration_count: int = 32
    run_mxq_compile: bool = False
    mxq_command_template: str = ''

cfg = RunConfig()
set_global_seed(cfg.seed)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
run_root = make_unique_run_root(DRIVE_RUNS_ROOT, cfg.run_slug)
checkpoints_dir = run_root / 'checkpoints'
exports_dir = run_root / 'exports'
calibration_dir = exports_dir / 'calibration'
for path in [run_root, checkpoints_dir, exports_dir, calibration_dir]:
    path.mkdir(parents=True, exist_ok=True)

print({'device': str(device), 'run_root': str(run_root), 'data_root': str(DRIVE_DATA_ROOT), 'config': asdict(cfg)})


{'device': 'cuda', 'run_root': '/content/drive/MyDrive/Data 255 Class Spring 2026/Data 255/Lab 3/runs/192606_2404_v5_clean_no_block_residual', 'data_root': '/content/drive/MyDrive/Data 255 Class Spring 2026/Data 255/Lab 3/Data', 'config': {'model_id': 'hinet_lite_npu_v6_v5_clean_no_block_residual', 'run_slug': 'v5_clean_no_block_residual', 'seed': 1337, 'eval_size': 256, 'crop_size': 128, 'batch_size': 8, 'epochs': 120, 'lr': 0.00025, 'min_lr': 1e-05, 'weight_decay': 5e-05, 'warmup_epochs': 5, 'grad_clip_norm': 1.0, 'mse_weight': 1.0, 'l1_weight': 0.1, 'num_workers': 2, 'use_amp': True, 'use_ema': True, 'ema_decay': 0.999, 'selected_weight_source': 'ema', 'train_pair_limit': None, 'val_pair_limit': None, 'calibration_count': 32, 'run_mxq_compile': False, 'mxq_command_template': ''}}


## 2. Data

Load paired L3 LR/HR images from Google Drive and confirm the Lab 3 image contract.

In [3]:
L3_TRAIN_SPLITS = (
    ('LR_HR/L3_HR_train1', 'L3_LR/L3_L3_train1', 'L3_HR_train1'),
    ('LR_HR/L3_HR_train2', 'L3_LR/L3_L3_train2', 'L3_HR_train2'),
    ('LR_HR/L3_HR_train3', 'L3_LR/L3_LR_train3', 'L3_HR_train3'),
    ('LR_HR/L3_HR_train4', 'L3_LR/L3_LR_train4', 'L3_HR_train4'),
    ('LR_HR/L3_HR_train', 'L3_LR/L3_LR_train', 'L3_HR_train'),
)
L3_VAL_DIRS = ('L3_HR_valid', 'L3_LR_valid')

def scan_image_directory(directory: Path) -> dict[str, Path]:
    if not directory.exists():
        raise FileNotFoundError(f'Missing directory: {directory}')
    return {path.stem: path for path in directory.iterdir() if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES}

def collect_split_pairs(hr_dir: Path, lr_dir: Path, split_name: str) -> list[tuple[Path, Path, str]]:
    hr_map = scan_image_directory(hr_dir)
    lr_map = scan_image_directory(lr_dir)
    pairs: list[tuple[Path, Path, str]] = []
    for stem in sorted(set(hr_map) & set(lr_map)):
        name = stem if split_name == 'val' else f'{split_name}/{stem}'
        pairs.append((lr_map[stem], hr_map[stem], name))
    return pairs

def collect_all_pairs(data_root: Path) -> tuple[list[tuple[Path, Path, str]], list[tuple[Path, Path, str]]]:
    train_pairs: list[tuple[Path, Path, str]] = []
    for hr_rel, lr_rel, split_name in L3_TRAIN_SPLITS:
        train_pairs.extend(collect_split_pairs(data_root / hr_rel, data_root / lr_rel, split_name))
    val_pairs = collect_split_pairs(data_root / L3_VAL_DIRS[0], data_root / L3_VAL_DIRS[1], 'val')
    return train_pairs, val_pairs

class PairedSRDataset(Dataset):
    def __init__(self, pairs: list[tuple[Path, Path, str]], *, train: bool, crop_size: int, eval_size: int, seed: int):
        self.pairs = pairs
        self.train = train
        self.crop_size = crop_size
        self.eval_size = eval_size
        self.seed = seed
        self.epoch = 0

    def set_epoch(self, epoch: int) -> None:
        self.epoch = epoch

    @staticmethod
    def to_tensor(image: Image.Image) -> torch.Tensor:
        array = np.asarray(image, dtype=np.float32) / 255.0
        return torch.from_numpy(array).permute(2, 0, 1).contiguous()

    def augment_pair(self, lr_image: Image.Image, hr_image: Image.Image, index: int) -> tuple[Image.Image, Image.Image]:
        rng = random.Random((self.seed * 1_000_003) + (self.epoch * 9_973) + index)
        width, height = lr_image.size
        if width < self.crop_size or height < self.crop_size:
            lr_image = lr_image.resize((self.eval_size, self.eval_size), BICUBIC_RESAMPLE)
            hr_image = hr_image.resize((self.eval_size, self.eval_size), BICUBIC_RESAMPLE)
            width, height = lr_image.size
        left = rng.randint(0, width - self.crop_size)
        top = rng.randint(0, height - self.crop_size)
        box = (left, top, left + self.crop_size, top + self.crop_size)
        lr_image = lr_image.crop(box)
        hr_image = hr_image.crop(box)
        if rng.random() < 0.5:
            lr_image = lr_image.transpose(FLIP_LEFT_RIGHT)
            hr_image = hr_image.transpose(FLIP_LEFT_RIGHT)
        if rng.random() < 0.5:
            lr_image = lr_image.transpose(FLIP_TOP_BOTTOM)
            hr_image = hr_image.transpose(FLIP_TOP_BOTTOM)
        rot = rng.choice([0, 1, 2, 3])
        if rot:
            rotate_ops = [ROTATE_90, ROTATE_180, ROTATE_270]
            lr_image = lr_image.transpose(rotate_ops[rot - 1])
            hr_image = hr_image.transpose(rotate_ops[rot - 1])
        return lr_image, hr_image

    def __len__(self) -> int:
        return len(self.pairs)

    def __getitem__(self, index: int) -> dict[str, Any]:
        lr_path, hr_path, name = self.pairs[index]
        lr_image = Image.open(lr_path).convert('RGB')
        hr_image = Image.open(hr_path).convert('RGB')
        if self.train:
            lr_image, hr_image = self.augment_pair(lr_image, hr_image, index)
        elif lr_image.size != (self.eval_size, self.eval_size):
            lr_image = lr_image.resize((self.eval_size, self.eval_size), BICUBIC_RESAMPLE)
            hr_image = hr_image.resize((self.eval_size, self.eval_size), BICUBIC_RESAMPLE)
        return {'lr': self.to_tensor(lr_image), 'hr': self.to_tensor(hr_image), 'name': name}

train_pairs, val_pairs = collect_all_pairs(DRIVE_DATA_ROOT)
if cfg.train_pair_limit is not None:
    train_pairs = train_pairs[: cfg.train_pair_limit]
if cfg.val_pair_limit is not None:
    val_pairs = val_pairs[: cfg.val_pair_limit]
if not train_pairs or not val_pairs:
    raise RuntimeError(f'Expected non-empty L3 train/val pairs, found train={len(train_pairs)} val={len(val_pairs)}')

train_ds = PairedSRDataset(train_pairs, train=True, crop_size=cfg.crop_size, eval_size=cfg.eval_size, seed=cfg.seed)
val_ds = PairedSRDataset(val_pairs, train=False, crop_size=cfg.eval_size, eval_size=cfg.eval_size, seed=cfg.seed)
loader_kwargs = {'num_workers': cfg.num_workers, 'pin_memory': device.type == 'cuda'}
if cfg.num_workers > 0:
    loader_kwargs['persistent_workers'] = True
train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, drop_last=False, **loader_kwargs)
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, drop_last=False, **loader_kwargs)
sample = val_ds[0]
print({'train_pairs': len(train_pairs), 'val_pairs': len(val_pairs), 'sample_lr_shape': tuple(sample['lr'].shape), 'sample_hr_shape': tuple(sample['hr'].shape)})


{'train_pairs': 2217, 'val_pairs': 110, 'sample_lr_shape': (3, 256, 256), 'sample_hr_shape': (3, 256, 256)}


## 3. Model and Training

Define and train `v5_clean_no_block_residual`: v5 width/depth, mixed kernels, no internal residual adds, U-Net skip adds retained, and global residual retained.

In [4]:
@dataclass
class HiNetLiteV5CleanConfig:
    channels: int = 44
    encoder_blocks: tuple[int, int] = (3, 3)
    bottleneck_blocks: int = 5
    decoder_blocks: tuple[int, int] = (3, 3)
    kernel_pattern: tuple[int, ...] = (3, 5, 3, 5, 3, 5)
    residual_scale: float = 0.2
    use_block_residual: bool = False
    use_unet_skips: bool = True
    global_residual: bool = True

def init_tail_small(conv: nn.Conv2d, scale: float = 1e-3) -> None:
    nn.init.kaiming_normal_(conv.weight, a=0.1, mode='fan_in', nonlinearity='leaky_relu')
    conv.weight.data.mul_(scale)
    if conv.bias is not None:
        nn.init.zeros_(conv.bias)

class MixedKernelCleanBlock(nn.Module):
    def __init__(self, channels: int, kernel_size: int, residual_scale: float, use_block_residual: bool):
        super().__init__()
        padding = kernel_size // 2
        self.residual_scale = float(residual_scale)
        self.use_block_residual = bool(use_block_residual)
        self.scale_folded = False
        self.conv1 = nn.Conv2d(channels, channels, kernel_size, 1, padding)
        self.act = nn.LeakyReLU(0.1, inplace=True)
        self.conv2 = nn.Conv2d(channels, channels, kernel_size, 1, padding)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = self.conv2(self.act(self.conv1(x)))
        if not self.use_block_residual:
            return residual
        if self.scale_folded:
            return x + residual
        return x + self.residual_scale * residual

    @torch.no_grad()
    def fold_residual_scale_for_export(self) -> None:
        if not self.use_block_residual or self.scale_folded:
            return
        self.conv2.weight.mul_(self.residual_scale)
        if self.conv2.bias is not None:
            self.conv2.bias.mul_(self.residual_scale)
        self.scale_folded = True
        self.residual_scale = 1.0

class DownsampleBlock(nn.Module):
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.block = nn.Sequential(nn.Conv2d(in_ch, out_ch, 3, stride=2, padding=1), nn.LeakyReLU(0.1, inplace=True))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)

class UpsampleBlock(nn.Module):
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.block = nn.Sequential(nn.ConvTranspose2d(in_ch, out_ch, 4, stride=2, padding=1), nn.LeakyReLU(0.1, inplace=True))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)

class HiNetLiteV5CleanNoBlockResidualSR(nn.Module):
    def __init__(self, model_cfg: HiNetLiteV5CleanConfig):
        super().__init__()
        c = model_cfg.channels
        pattern = model_cfg.kernel_pattern
        self.use_unet_skips = model_cfg.use_unet_skips
        self.global_residual = model_cfg.global_residual
        self.stem = nn.Sequential(nn.Conv2d(3, c, 3, 1, 1), nn.LeakyReLU(0.1, inplace=True))
        self.enc1 = nn.Sequential(*[
            MixedKernelCleanBlock(c, pattern[idx % len(pattern)], model_cfg.residual_scale, model_cfg.use_block_residual)
            for idx in range(model_cfg.encoder_blocks[0])
        ])
        self.down1 = DownsampleBlock(c, c * 2)
        self.enc2 = nn.Sequential(*[
            MixedKernelCleanBlock(c * 2, pattern[(model_cfg.encoder_blocks[0] + idx) % len(pattern)], model_cfg.residual_scale, model_cfg.use_block_residual)
            for idx in range(model_cfg.encoder_blocks[1])
        ])
        self.down2 = DownsampleBlock(c * 2, c * 4)
        self.bottleneck = nn.Sequential(*[
            MixedKernelCleanBlock(c * 4, pattern[(sum(model_cfg.encoder_blocks) + idx) % len(pattern)], model_cfg.residual_scale, model_cfg.use_block_residual)
            for idx in range(model_cfg.bottleneck_blocks)
        ])
        self.up1 = UpsampleBlock(c * 4, c * 2)
        self.dec1 = nn.Sequential(*[
            MixedKernelCleanBlock(c * 2, pattern[(sum(model_cfg.encoder_blocks) + model_cfg.bottleneck_blocks + idx) % len(pattern)], model_cfg.residual_scale, model_cfg.use_block_residual)
            for idx in range(model_cfg.decoder_blocks[0])
        ])
        self.up2 = UpsampleBlock(c * 2, c)
        self.dec2 = nn.Sequential(*[
            MixedKernelCleanBlock(c, pattern[(sum(model_cfg.encoder_blocks) + model_cfg.bottleneck_blocks + model_cfg.decoder_blocks[0] + idx) % len(pattern)], model_cfg.residual_scale, model_cfg.use_block_residual)
            for idx in range(model_cfg.decoder_blocks[1])
        ])
        self.tail = nn.Conv2d(c, 3, 3, 1, 1)
        init_tail_small(self.tail)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        stem = self.stem(x)
        enc1 = self.enc1(stem)
        enc2 = self.enc2(self.down1(enc1))
        bottleneck = self.bottleneck(self.down2(enc2))
        up1 = self.up1(bottleneck)
        dec1_input = up1 + enc2 if self.use_unet_skips else up1
        dec1 = self.dec1(dec1_input)
        up2 = self.up2(dec1)
        dec2_input = up2 + enc1 if self.use_unet_skips else up2
        dec2 = self.dec2(dec2_input)
        delta = self.tail(dec2)
        return x + delta if self.global_residual else delta

    @torch.no_grad()
    def fold_residual_scales_for_export(self) -> None:
        for module in self.modules():
            if isinstance(module, MixedKernelCleanBlock):
                module.fold_residual_scale_for_export()

def count_parameters(module: nn.Module) -> dict[str, int]:
    total = sum(param.numel() for param in module.parameters())
    trainable = sum(param.numel() for param in module.parameters() if param.requires_grad)
    return {'total': int(total), 'trainable': int(trainable)}

model_cfg = HiNetLiteV5CleanConfig()
model = HiNetLiteV5CleanNoBlockResidualSR(model_cfg).to(device)
with torch.inference_mode():
    shape_output = model(torch.randn(1, 3, 256, 256, device=device))
if tuple(shape_output.shape) != (1, 3, 256, 256):
    raise RuntimeError(f'Unexpected model output shape: {tuple(shape_output.shape)}')
model_num_parameters = count_parameters(model)
print({'model_name': model.__class__.__name__, 'model_config': asdict(model_cfg), 'shape': tuple(shape_output.shape), 'parameter_count': model_num_parameters})


{'model_name': 'HiNetLiteV5CleanNoBlockResidualSR', 'model_config': {'channels': 44, 'encoder_blocks': (3, 3), 'bottleneck_blocks': 5, 'decoder_blocks': (3, 3), 'kernel_pattern': (3, 5, 3, 5, 3, 5), 'residual_scale': 0.2, 'use_block_residual': False, 'use_unet_skips': True, 'global_residual': True}, 'shape': (1, 3, 256, 256), 'parameter_count': {'total': 7421043, 'trainable': 7421043}}


In [5]:
def psnr_from_mse(mse: torch.Tensor, eps: float = 1e-12) -> torch.Tensor:
    return 10.0 * torch.log10(1.0 / torch.clamp(mse, min=eps))

@torch.no_grad()
def batch_psnr(pred: torch.Tensor, target: torch.Tensor) -> float:
    mse = F.mse_loss(pred, target, reduction='none').mean(dim=(1, 2, 3))
    return float(psnr_from_mse(mse).mean().item())

def move_image_batch(batch: torch.Tensor) -> torch.Tensor:
    return batch.to(device, non_blocking=(device.type == 'cuda'))

def restoration_loss(pred: torch.Tensor, lr: torch.Tensor, hr: torch.Tensor) -> dict[str, torch.Tensor]:
    pred_res = pred - lr
    target_res = hr - lr
    loss_mse = F.mse_loss(pred_res, target_res)
    loss_l1 = F.l1_loss(pred_res, target_res)
    loss = cfg.mse_weight * loss_mse + cfg.l1_weight * loss_l1
    return {'loss': loss, 'loss_mse': loss_mse.detach(), 'loss_l1': loss_l1.detach()}

def lr_at_epoch(epoch_idx: int) -> float:
    step = epoch_idx + 1
    if cfg.warmup_epochs > 0 and step <= cfg.warmup_epochs:
        return cfg.lr * step / max(1, cfg.warmup_epochs)
    progress = max(0.0, (step - cfg.warmup_epochs) / max(1, cfg.epochs - cfg.warmup_epochs))
    cosine = 0.5 * (1.0 + math.cos(math.pi * min(progress, 1.0)))
    return cfg.min_lr + cosine * (cfg.lr - cfg.min_lr)

def autocast_context(enabled: bool):
    if not enabled:
        return nullcontext()
    if hasattr(torch, 'amp') and hasattr(torch.amp, 'autocast'):
        return torch.amp.autocast(device_type=device.type, enabled=enabled)
    return torch.cuda.amp.autocast(enabled=enabled)

def make_grad_scaler(enabled: bool):
    if hasattr(torch, 'amp') and hasattr(torch.amp, 'GradScaler'):
        return torch.amp.GradScaler('cuda', enabled=enabled)
    return torch.cuda.amp.GradScaler(enabled=enabled)

class EMA:
    def __init__(self, source_model: nn.Module, decay: float):
        self.decay = decay
        self.shadow = {key: value.detach().clone() for key, value in source_model.state_dict().items()}

    @torch.no_grad()
    def update(self, source_model: nn.Module) -> None:
        for key, value in source_model.state_dict().items():
            self.shadow[key].mul_(self.decay).add_(value.detach(), alpha=1.0 - self.decay)

    @torch.no_grad()
    def copy_to(self, target_model: nn.Module) -> None:
        target_model.load_state_dict(self.shadow, strict=True)

@torch.inference_mode()
def evaluate_psnr(eval_model: nn.Module, loader: DataLoader) -> dict[str, float]:
    eval_model.eval()
    pred_meter = 0.0
    input_meter = 0.0
    count = 0
    for batch in loader:
        lr = move_image_batch(batch['lr'])
        hr = move_image_batch(batch['hr'])
        pred = eval_model(lr).clamp(0, 1)
        pred_meter += batch_psnr(pred, hr)
        input_meter += batch_psnr(lr, hr)
        count += 1
    return {'val_psnr': pred_meter / max(count, 1), 'input_psnr': input_meter / max(count, 1), 'delta_psnr': (pred_meter - input_meter) / max(count, 1)}

def train_one_epoch(epoch: int, optimizer: torch.optim.Optimizer, scaler: Any, ema: EMA | None) -> dict[str, float]:
    model.train()
    train_ds.set_epoch(epoch)
    amp_enabled = bool(cfg.use_amp and device.type == 'cuda')
    loss_meter = 0.0
    psnr_meter = 0.0
    count = 0
    for batch in train_loader:
        lr = move_image_batch(batch['lr'])
        hr = move_image_batch(batch['hr'])
        optimizer.zero_grad(set_to_none=True)
        with autocast_context(amp_enabled):
            pred = model(lr)
            loss_info = restoration_loss(pred, lr, hr)
            loss = loss_info['loss']
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip_norm)
        scaler.step(optimizer)
        scaler.update()
        if ema is not None:
            ema.update(model)
        with torch.no_grad():
            loss_meter += float(loss.item())
            psnr_meter += batch_psnr(pred.clamp(0, 1), hr)
            count += 1
    return {'train_loss': loss_meter / max(count, 1), 'train_psnr': psnr_meter / max(count, 1)}

def build_eval_model(ema: EMA | None) -> nn.Module:
    eval_model = HiNetLiteV5CleanNoBlockResidualSR(model_cfg).to(device).eval()
    if cfg.selected_weight_source == 'ema' and ema is not None:
        ema.copy_to(eval_model)
    else:
        eval_model.load_state_dict(model.state_dict(), strict=True)
    return eval_model

def save_checkpoint(path: Path, epoch: int, optimizer: torch.optim.Optimizer, scaler: Any, ema: EMA | None, metrics: dict[str, float], best_val_psnr: float, best_epoch: int) -> None:
    payload = {
        'epoch': epoch,
        'model': model.state_dict(),
        'ema_model': ema.shadow if ema is not None else None,
        'optimizer': optimizer.state_dict(),
        'scaler': scaler.state_dict() if scaler.is_enabled() else None,
        'config': asdict(cfg),
        'model_config': asdict(model_cfg),
        'selected_weight_source': cfg.selected_weight_source,
        'best_val_psnr': float(best_val_psnr),
        'best_epoch': int(best_epoch),
        'last_metrics': metrics,
    }
    torch.save(payload, path)

optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
scaler = make_grad_scaler(enabled=(cfg.use_amp and device.type == 'cuda'))
ema = EMA(model, cfg.ema_decay) if cfg.use_ema else None
best_ckpt_path = checkpoints_dir / 'best.pth'
last_ckpt_path = checkpoints_dir / 'last.pth'
metrics_csv_path = run_root / 'metrics.csv'
history_fields = ['epoch', 'lr', 'train_loss', 'train_psnr', 'val_psnr', 'input_psnr', 'delta_psnr']
best_val_psnr = float('-inf')
best_epoch = 0

with metrics_csv_path.open('w', newline='', encoding='utf-8') as handle:
    writer = csv.DictWriter(handle, fieldnames=history_fields)
    writer.writeheader()

for epoch in range(1, cfg.epochs + 1):
    epoch_lr = lr_at_epoch(epoch - 1)
    for group in optimizer.param_groups:
        group['lr'] = epoch_lr
    train_metrics = train_one_epoch(epoch, optimizer, scaler, ema)
    eval_model = build_eval_model(ema)
    val_metrics = evaluate_psnr(eval_model, val_loader)
    row = {'epoch': epoch, 'lr': epoch_lr, **train_metrics, **val_metrics}
    is_best = val_metrics['val_psnr'] > best_val_psnr
    if is_best:
        best_val_psnr = float(val_metrics['val_psnr'])
        best_epoch = epoch
        save_checkpoint(best_ckpt_path, epoch, optimizer, scaler, ema, row, best_val_psnr, best_epoch)
    save_checkpoint(last_ckpt_path, epoch, optimizer, scaler, ema, row, best_val_psnr, best_epoch)
    with metrics_csv_path.open('a', newline='', encoding='utf-8') as handle:
        writer = csv.DictWriter(handle, fieldnames=history_fields)
        writer.writerow({key: row[key] for key in history_fields})
    print({'epoch': epoch, 'lr': epoch_lr, 'train_loss': train_metrics['train_loss'], 'train_psnr': train_metrics['train_psnr'], 'val_psnr': val_metrics['val_psnr'], 'input_psnr': val_metrics['input_psnr'], 'best_val_psnr': best_val_psnr})

print({'best_checkpoint': str(best_ckpt_path), 'best_epoch': best_epoch, 'best_val_psnr': best_val_psnr})


{'epoch': 1, 'lr': 5e-05, 'train_loss': 0.00965634692399997, 'train_psnr': 24.460571371394096, 'val_psnr': 23.134069460088558, 'input_psnr': 23.133060299266468, 'best_val_psnr': 23.134069460088558}
{'epoch': 2, 'lr': 0.0001, 'train_loss': 0.009596688805228407, 'train_psnr': 24.499210515468242, 'val_psnr': 23.13550291928378, 'input_psnr': 23.133060299266468, 'best_val_psnr': 23.13550291928378}
{'epoch': 3, 'lr': 0.00015000000000000001, 'train_loss': 0.00939935252150829, 'train_psnr': 24.629405694042177, 'val_psnr': 23.136647328463468, 'input_psnr': 23.133060299266468, 'best_val_psnr': 23.136647328463468}
{'epoch': 4, 'lr': 0.0002, 'train_loss': 0.009348284902064491, 'train_psnr': 24.677808329355802, 'val_psnr': 23.138963595303622, 'input_psnr': 23.133060299266468, 'best_val_psnr': 23.138963595303622}
{'epoch': 5, 'lr': 0.00025, 'train_loss': 0.009243218866411409, 'train_psnr': 24.725435071711917, 'val_psnr': 23.144978974082253, 'input_psnr': 23.133060299266468, 'best_val_psnr': 23.14497

## 4. Validation

Reload the selected checkpoint and report validation PSNR against the L3 validation set.

In [6]:
def load_best_eval_model(checkpoint_path: Path, runtime_device: torch.device) -> nn.Module:
    checkpoint = torch.load(checkpoint_path, map_location=runtime_device)
    loaded_model_cfg = HiNetLiteV5CleanConfig(**checkpoint.get('model_config', asdict(model_cfg)))
    eval_model = HiNetLiteV5CleanNoBlockResidualSR(loaded_model_cfg).to(runtime_device).eval()
    source_state = checkpoint.get('ema_model') if checkpoint.get('selected_weight_source') == 'ema' else checkpoint.get('model')
    if source_state is None:
        source_state = checkpoint['model']
    eval_model.load_state_dict(source_state, strict=True)
    return eval_model

best_eval_model = load_best_eval_model(best_ckpt_path, device)
validation_summary = evaluate_psnr(best_eval_model, val_loader)
print({'validation_summary': validation_summary, 'best_checkpoint': str(best_ckpt_path), 'best_epoch': best_epoch})


{'validation_summary': {'val_psnr': 23.930102400346236, 'input_psnr': 23.133060299266468, 'delta_psnr': 0.7970421010797674}, 'best_checkpoint': '/content/drive/MyDrive/Data 255 Class Spring 2026/Data 255/Lab 3/runs/192606_2404_v5_clean_no_block_residual/checkpoints/best.pth', 'best_epoch': 81}


## 5. Export and Submission Artifacts

Export fixed-shape ONNX, check graph operators, create training-derived calibration images, and record the MXQ handoff.

In [7]:
SUSPICIOUS_OPS = {
    'Mul', 'Add', 'Resize', 'Concat', 'MatMul', 'Softmax', 'Sigmoid', 'Tanh',
    'InstanceNormalization', 'GroupNormalization', 'LayerNormalization', 'ReduceMean',
    'Div', 'Sub', 'Pow', 'Sqrt', 'Reshape', 'Transpose', 'Slice', 'Gather', 'Unsqueeze', 'Squeeze',
}

def audit_onnx_graph(onnx_path: Path) -> dict[str, Any]:
    if onnx is None:
        return {'checked': False, 'reason': 'onnx unavailable'}
    loaded = onnx.load(str(onnx_path))
    onnx.checker.check_model(loaded)
    ops = [node.op_type for node in loaded.graph.node]
    counts = dict(sorted(Counter(ops).items()))
    suspicious = {op: counts[op] for op in sorted(SUSPICIOUS_OPS) if op in counts}
    return {
        'checked': True,
        'onnx_checker': 'passed',
        'total_node_count': len(ops),
        'unique_op_types': sorted(counts),
        'op_counts': counts,
        'suspicious_ops_found': suspicious,
        'mul_found': 'Mul' in counts,
        'add_found': 'Add' in counts,
    }

export_model = load_best_eval_model(best_ckpt_path, device)
export_model.eval()
fold_input = torch.randn(1, 3, 256, 256, device=device)
with torch.inference_mode():
    before_fold = export_model(fold_input).detach()
export_model.fold_residual_scales_for_export()
with torch.inference_mode():
    after_fold = export_model(fold_input).detach()
folding_summary = {
    'fold_max_abs_diff': float((before_fold - after_fold).abs().max().item()),
    'fold_mean_abs_diff': float((before_fold - after_fold).abs().mean().item()),
}
print(folding_summary)

onnx_path = exports_dir / f'{cfg.model_id}.onnx'
dummy = torch.randn(1, 3, 256, 256, device=device)
torch.onnx.export(
    export_model,
    dummy,
    str(onnx_path),
    input_names=['input'],
    output_names=['output'],
    opset_version=13,
    do_constant_folding=True,
    dynamic_axes=None,
    dynamo=False,
)
op_audit = audit_onnx_graph(onnx_path)
print({'onnx_path': str(onnx_path), **op_audit})

parity_result = {'attempted': False, 'onnxruntime_available': ort is not None, 'passed': False, 'max_abs_diff': None, 'mean_abs_diff': None}
if ort is not None:
    cpu_model = load_best_eval_model(best_ckpt_path, torch.device('cpu')).cpu().eval()
    cpu_model.fold_residual_scales_for_export()
    parity_input = torch.randn(1, 3, 256, 256)
    with torch.no_grad():
        torch_out = cpu_model(parity_input).detach().cpu().numpy()
    session = ort.InferenceSession(str(onnx_path), providers=['CPUExecutionProvider'])
    ort_out = session.run(['output'], {'input': parity_input.numpy()})[0]
    diff = np.abs(torch_out - ort_out)
    parity_result.update({'attempted': True, 'passed': bool(float(diff.max()) < 1e-3), 'max_abs_diff': float(diff.max()), 'mean_abs_diff': float(diff.mean())})
print(parity_result)


{'fold_max_abs_diff': 0.0, 'fold_mean_abs_diff': 0.0}


/tmp/ipykernel_46051/1142547989.py:42: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


{'onnx_path': '/content/drive/MyDrive/Data 255 Class Spring 2026/Data 255/Lab 3/runs/192606_2404_v5_clean_no_block_residual/exports/hinet_lite_npu_v6_v5_clean_no_block_residual.onnx', 'checked': True, 'onnx_checker': 'passed', 'total_node_count': 65, 'unique_op_types': ['Add', 'Conv', 'ConvTranspose', 'LeakyRelu'], 'op_counts': {'Add': 3, 'Conv': 38, 'ConvTranspose': 2, 'LeakyRelu': 22}, 'suspicious_ops_found': {'Add': 3}, 'mul_found': False, 'add_found': True}
{'attempted': True, 'onnxruntime_available': True, 'passed': True, 'max_abs_diff': 9.5367431640625e-07, 'mean_abs_diff': 1.498347330652905e-07}


In [8]:
def write_calibration_bundle(train_pairs: list[tuple[Path, Path, str]], out_dir: Path, max_items: int) -> dict[str, Any]:
    out_dir.mkdir(parents=True, exist_ok=True)
    selected = train_pairs[: max_items]
    manifest = {'source': 'L3 training LR images', 'derived_from_training': True, 'count': len(selected), 'items': []}
    for idx, (lr_path, _, name) in enumerate(selected):
        dst = out_dir / f'cal_{idx:04d}.png'
        image = Image.open(lr_path).convert('RGB')
        if image.size != (cfg.eval_size, cfg.eval_size):
            image = image.resize((cfg.eval_size, cfg.eval_size), BICUBIC_RESAMPLE)
        image.save(dst)
        manifest['items'].append({'index': idx, 'name': name, 'lr_source': str(lr_path), 'calibration_path': str(dst)})
    manifest_path = out_dir / 'manifest.json'
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding='utf-8')
    return {'calibration_dir': str(out_dir), 'manifest_path': str(manifest_path), 'count': len(selected), 'derived_from_training': True}

conversion_script_path = exports_dir / 'convert_v5_clean_mxq.py'
conversion_script = r'''#!/usr/bin/env python3
from __future__ import annotations
import argparse, json, shlex, shutil, subprocess
from pathlib import Path

def find_tool():
    for name in ['mxq_compile', 'qubee', 'qb']:
        path = shutil.which(name)
        if path:
            return path
    return None

parser = argparse.ArgumentParser(description='ONNX to MXQ helper for HiNetLite v6.')
parser.add_argument('--onnx', type=Path, required=True)
parser.add_argument('--calibration-dir', type=Path, required=True)
parser.add_argument('--output', type=Path, required=True)
parser.add_argument('--command-template', default='')
parser.add_argument('--dry-run', action='store_true')
args = parser.parse_args()
args.onnx = args.onnx.resolve()
args.calibration_dir = args.calibration_dir.resolve()
args.output = args.output.resolve()
if not args.onnx.exists():
    raise FileNotFoundError(args.onnx)
if not args.calibration_dir.exists():
    raise FileNotFoundError(args.calibration_dir)
tool = find_tool()
payload = {'onnx': str(args.onnx), 'calibration_dir': str(args.calibration_dir), 'output': str(args.output), 'mxq_tool_detected': tool, 'status': 'handoff_only'}
if args.command_template:
    command = args.command_template.format(onnx=shlex.quote(str(args.onnx)), calibration=shlex.quote(str(args.calibration_dir)), output=shlex.quote(str(args.output)))
elif tool:
    command = ' '.join([shlex.quote(tool), shlex.quote(str(args.onnx)), shlex.quote(str(args.calibration_dir)), shlex.quote(str(args.output))])
else:
    print(json.dumps(payload, indent=2))
    raise SystemExit(0)
payload['command'] = command
payload['status'] = 'dry_run' if args.dry_run else 'running'
if args.dry_run:
    print(json.dumps(payload, indent=2))
    raise SystemExit(0)
completed = subprocess.run(command, shell=True, capture_output=True, text=True)
payload.update({'returncode': completed.returncode, 'stdout': completed.stdout, 'stderr': completed.stderr, 'status': 'completed' if completed.returncode == 0 else 'failed', 'output_exists': args.output.exists()})
print(json.dumps(payload, indent=2))
raise SystemExit(completed.returncode)
'''
conversion_script_path.write_text(conversion_script, encoding='utf-8')
calibration_info = write_calibration_bundle(train_pairs, calibration_dir, cfg.calibration_count)
mxq_target_path = exports_dir / f'{cfg.model_id}.mxq'
mxq_command = [sys.executable, str(conversion_script_path), '--onnx', str(onnx_path), '--calibration-dir', str(calibration_dir), '--output', str(mxq_target_path)]
if cfg.mxq_command_template:
    mxq_command.extend(['--command-template', cfg.mxq_command_template])
if not cfg.run_mxq_compile:
    mxq_command.append('--dry-run')
mxq_handoff = {
    'conversion_script_path': str(conversion_script_path),
    'onnx_input_path': str(onnx_path),
    'calibration_manifest_path': calibration_info['manifest_path'],
    'mxq_target_path': str(mxq_target_path),
    'command': ' '.join(shlex.quote(part) for part in mxq_command),
    'requested_compile': cfg.run_mxq_compile,
}
print({'calibration': calibration_info, 'mxq_handoff': mxq_handoff})


{'calibration': {'calibration_dir': '/content/drive/MyDrive/Data 255 Class Spring 2026/Data 255/Lab 3/runs/192606_2404_v5_clean_no_block_residual/exports/calibration', 'manifest_path': '/content/drive/MyDrive/Data 255 Class Spring 2026/Data 255/Lab 3/runs/192606_2404_v5_clean_no_block_residual/exports/calibration/manifest.json', 'count': 32, 'derived_from_training': True}, 'mxq_handoff': {'conversion_script_path': '/content/drive/MyDrive/Data 255 Class Spring 2026/Data 255/Lab 3/runs/192606_2404_v5_clean_no_block_residual/exports/convert_v5_clean_mxq.py', 'onnx_input_path': '/content/drive/MyDrive/Data 255 Class Spring 2026/Data 255/Lab 3/runs/192606_2404_v5_clean_no_block_residual/exports/hinet_lite_npu_v6_v5_clean_no_block_residual.onnx', 'calibration_manifest_path': '/content/drive/MyDrive/Data 255 Class Spring 2026/Data 255/Lab 3/runs/192606_2404_v5_clean_no_block_residual/exports/calibration/manifest.json', 'mxq_target_path': '/content/drive/MyDrive/Data 255 Class Spring 2026/Data

## 6. Final Summary

Write the final JSON and Markdown summaries with paths for `.pth`, `.onnx`, calibration, conversion script, and `.mxq` target.

In [9]:
summary = {
    'model_id': cfg.model_id,
    'variant': cfg.run_slug,
    'run_root': str(run_root),
    'config': asdict(cfg),
    'model_config': asdict(model_cfg),
    'parameter_count': model_num_parameters,
    'validation': validation_summary,
    'best_epoch': int(best_epoch),
    'best_val_psnr': float(best_val_psnr),
    'best_checkpoint_path': str(best_ckpt_path),
    'last_checkpoint_path': str(last_ckpt_path),
    'onnx_path': str(onnx_path),
    'folding_summary': folding_summary,
    'onnx_parity': parity_result,
    'op_audit': op_audit,
    'calibration': calibration_info,
    'mxq_handoff': mxq_handoff,
    'artifact_paths': {
        'pth': str(best_ckpt_path),
        'onnx': str(onnx_path),
        'calibration_manifest': calibration_info['manifest_path'],
        'conversion_script': str(conversion_script_path),
        'mxq_target': str(mxq_target_path),
    },
}
summary_path = run_root / 'summary.json'
summary_path.write_text(json.dumps(summary, indent=2), encoding='utf-8')
report_path = run_root / 'v5_clean_experiment_summary.md'
report_text = f'''# HiNetLite NPU v6 - v5_clean_no_block_residual

| Field | Value |
| --- | --- |
| Variant | `{cfg.run_slug}` |
| Parameter count | `{model_num_parameters['total']}` |
| Best validation PSNR | `{best_val_psnr:.4f}` |
| Input baseline PSNR | `{validation_summary['input_psnr']:.4f}` |
| Best checkpoint | `{best_ckpt_path}` |
| ONNX path | `{onnx_path}` |
| ONNX node count | `{op_audit.get('total_node_count')}` |
| Suspicious ONNX ops | `{op_audit.get('suspicious_ops_found')}` |
| MXQ target | `{mxq_target_path}` |
| NPU latency | `not available` |

Recommendation: continue only if MXQ conversion succeeds, `Mul` is absent, and measured NPU latency is acceptable with the remaining `Add` ops.
'''
report_path.write_text(report_text, encoding='utf-8')
print({'summary_path': str(summary_path), 'report_path': str(report_path), 'artifacts': summary['artifact_paths']})


{'summary_path': '/content/drive/MyDrive/Data 255 Class Spring 2026/Data 255/Lab 3/runs/192606_2404_v5_clean_no_block_residual/summary.json', 'report_path': '/content/drive/MyDrive/Data 255 Class Spring 2026/Data 255/Lab 3/runs/192606_2404_v5_clean_no_block_residual/v5_clean_experiment_summary.md', 'artifacts': {'pth': '/content/drive/MyDrive/Data 255 Class Spring 2026/Data 255/Lab 3/runs/192606_2404_v5_clean_no_block_residual/checkpoints/best.pth', 'onnx': '/content/drive/MyDrive/Data 255 Class Spring 2026/Data 255/Lab 3/runs/192606_2404_v5_clean_no_block_residual/exports/hinet_lite_npu_v6_v5_clean_no_block_residual.onnx', 'calibration_manifest': '/content/drive/MyDrive/Data 255 Class Spring 2026/Data 255/Lab 3/runs/192606_2404_v5_clean_no_block_residual/exports/calibration/manifest.json', 'conversion_script': '/content/drive/MyDrive/Data 255 Class Spring 2026/Data 255/Lab 3/runs/192606_2404_v5_clean_no_block_residual/exports/convert_v5_clean_mxq.py', 'mxq_target': '/content/drive/MyD